# Рекомендация тарифов

## Подготовка данных

### Загрузка библиотек и датасета

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier 
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
data = pd.read_csv('/datasets/users_behavior.csv')

In [3]:
data.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB



Каждый объект в наборе данных — это информация о поведении одного пользователя за месяц:    

* сalls — количество звонков,  
* minutes — суммарная длительность звонков в минутах,  
* messages — количество sms-сообщений,  
* mb_used — израсходованный интернет-трафик в Мб,  
* is_ultra — каким тарифом пользовался в течение месяца («Ультра» — 1, «Смарт» — 0).  

## Исследование задачи

### Разделение исходных данных на обучающую, валидационную и тестовую выборки

Для того, чтобы разделить исходные данные data на 3 части: обучающую, валидационную и тестовую, используем функцию train_test_split библиотеки sklearn. Будем делить данные в соотношении 3:1:1.  

Валидационная выборка должна составлять 25% исходных данных. Чтобы получить валидационную выборку размером 25%, нужно разделить исходные данные на две части: обучающую и временную выборку (включая в себя валидационную и тестовую выборки). Затем, временную выборку разделить пополам, чтобы получить валидационную и тестовую выборки равного размера.  

In [5]:
# Разделим исходные данные на обучающую и временную выборку
train_data, temp_data = train_test_split(data, test_size=0.4, random_state=12345)

In [6]:
# Разделим временную выборку на валидационную и тестовую выборки
valid_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=12345)

Создадим отдельные датасеты с целевыми и прочими признаками для выборок.
За **целевой признак** возьмем значения из столбца **is_ultra** (каким тарифом пользовался в течение месяца пользователь: «Ультра» — 1, «Смарт» — 0). Остальные признаки будем использовать для предсказаний.

In [7]:
# Признаки для обучающей выборки
train_features = train_data.drop('is_ultra', axis=1)
# Целевой признак для обучающей выборки
train_target = train_data['is_ultra']

# Признаки для валидационной выборки
valid_features = valid_data.drop('is_ultra', axis=1)
# Целевой признак для валидационной выборки
valid_target = valid_data['is_ultra']

# Признаки для тестовой выборки
test_features = test_data.drop('is_ultra', axis=1)
# Целевой признак для тестовой выборки
test_target = test_data['is_ultra']

In [8]:
# Проверим размеры выборок
print("Размер обучающей выборки:", train_data.shape)
print("Размер валидационной выборки:", valid_data.shape)
print("Размер тестовой выборки:", test_data.shape)

Размер обучающей выборки: (1928, 5)
Размер валидационной выборки: (643, 5)
Размер тестовой выборки: (643, 5)


**Вывод**

* На первом этапе исследования мы разбили исходную выборку на 3 части в соотношении 3:1:1.  
* Определили целевой признак (is_ultra) и создали отдельные датасеты:  
1) Обучение пройдет на данных из датасета train_data  
2) Валидация модели на данных из датасета valid_data  
3) Лучшая модель по валидации будет применена на данных датасета test_data  

## Исследование моделей

В нашем исследовании стоит задача **Классификации** (точнее бинарной/двоичной классификации), так как наш целевой признак is_ultra - категориальный (каким тарифом пользовался в течение месяца пользователь: «Ультра» — 1, «Смарт» — 0).

Таким образом, мы исследуем три модели Классификации:

* Решающее дерево  
* Случайный лес    
* Логистическую регрессию  

### Модель: Решающее дерево

Для начала исследуем модель **Решающего дерева (DecisionTreeClassifier)**. Самый важный гиперпараметр решающего дерева **max_depth** (максимальная глубина). Построим алгоритм, который перебирает значения max_depth в диапазоне от 1 до 5 и сохраняет модель с лучшим значением метрики accuracy.

In [9]:
best_tree_model = None
best_result = 0
best_depth = 0

for depth in range(1, 6):
    best_tree_model = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    best_tree_model.fit(train_features, train_target) #обучаем модель
    predictions_valid = best_tree_model.predict(valid_features) #получаем предсказания модели
    result = accuracy_score(valid_target, predictions_valid) #посчитаем качество модели accuracy
    if result > best_result:
        best_tree_model = best_tree_model #сохраним наилучшую модель
        best_result = result #сохраним наилучшее значение метрики accuracy
        best_depth = depth 
        
print("Accuracy лучшей модели:", best_result)
print('Глубина для лучшей модели:', best_depth)

Accuracy лучшей модели: 0.7853810264385692
Глубина для лучшей модели: 3


Вывод:  
Accuracy лучшей модели решающего дерева: 0.78, лучшая глубина (max_depth) для такой модели - 3.

### Модель: Случайный лес

Перейдем к оценке модели **Случайного леса (RandomForestClassifier)**. Важнейший гиперпараметр этой модели **n_estimators** («количество оценщиков»). Построим алгоритм, который перебирает значения n_estimators в диапазоне от 1 до 3, max_depth в диапазоне от 1 до 5 и сохраняет модель с лучшим значением метрики accuracy.

In [10]:
best_forest_model = None
best_result = 0
best_estim = 0
best_depth = 0

for est in range(1, 4):
    for depth in range (1, 6):
        best_forest_model = RandomForestClassifier(random_state=12345, n_estimators=est)
        best_forest_model.fit(train_features, train_target) #обучаем модель
        predictions_valid = best_forest_model.predict(valid_features) #получим предсказания модели
        result = accuracy_score(valid_target, predictions_valid) #посчитаем качество модели accuracy
    if result > best_result:
        best_forest_model = best_forest_model #сохраним наилучшую модель
        best_result = result #сохраним наилучшее значение метрики accuracy
        best_estim = est 
        best_depth = depth

print("Accuracy наилучшей модели на валидационной выборке:", best_result)
print('Лучшая количество деревьев', best_estim)
print('Глубина для лучшей модели:', best_depth)

Accuracy наилучшей модели на валидационной выборке: 0.7636080870917574
Лучшая количество деревьев 2
Глубина для лучшей модели: 5


Вывод:  
Accuracy лучшей модели случайного леса: 0.764, лучшая глубина (max_depth) для такой модели - 5, Лучшая количество деревьев (n_estimators) - 2.

### Модель: Логистическая регрессия

Протестируем последнюю модель классификации **Логистическую регрессию (LogisticRegression)**. Изменим гиперпараметр **max_iter** - 1000, который задаёт максимальное количество итераций обучения. Добавим дополнительный гиперпараметр - **solver='lbfgs'** (алгоритм 'lbfgs' — один из самых распространённых и подходит для большинства задач).

In [11]:
best_reg_model = LogisticRegression(random_state=12345, solver='lbfgs', max_iter=1000)
best_reg_model.fit(train_features, train_target)
valid_predictions = best_reg_model.predict(valid_features)
accuracy = accuracy_score(valid_target, predictions_valid)

print('Accuracy логистической регрессии:', accuracy)

Accuracy логистической регрессии: 0.7387247278382582


**Выводы:**

В ходе исследования тестировались 3 модели классификации. Были получены следующие результаты:  

* Точность (accuracy) модели Решающего дерева: 0.785, при max_depth равной 3.  
* Точность (accuracy) модели Случайного леса: 0.764, при max_depth равной 5 и n_estimators равному 2.  
* Точность (accuracy) модели Логистической регресии: 0.779  

Таким образом, лучший результат показывает модель Решающего дерева: 0.785 (max_depth = 3). Проверим качество этой модели на тестовой выборке.

## Проверка модели на тестовой выборке

Модель Решающего дерева показала наилучший результат. Теперь проверим точность этой модели на тестовой выборке test_data. Наша цель: довести долю правильных ответов по крайней мере до 0.75.  

Создадим решающее дерево с максимальной глубиной 3 и обучим его на всей тренировочной выборке (включая данные из валидационной выборки). Затем, мы проверим accuracy на тестовой выборке test_data, чтобы убедиться, что доля правильных ответов превышает 0.75.  

In [12]:
# Модель Решающего дерева с максимальной глубиной 3
best_tree_model = DecisionTreeClassifier(random_state=12345, max_depth=3)

In [13]:
# Объединим тренировочную и валидационную выборку в одну combined_train_features
combined_train_features = pd.concat([train_features, valid_features])
combined_train_target = pd.concat([train_target, valid_target])

In [14]:
# Обучим модель Решающего дерева на объединенной обучающей выборке combined_train_features
best_tree_model.fit(combined_train_features, combined_train_target)

DecisionTreeClassifier(max_depth=3, random_state=12345)

In [15]:
# Получим предсказания на тестовой выборке test_features
predictions_test = best_tree_model.predict(test_features)

In [16]:
# Получим accuracy на тестовой выборке
accuracy_test = accuracy_score(test_target, predictions_test)
print("Accuracy на тестовой выборке:", accuracy_test)

Accuracy на тестовой выборке: 0.776049766718507


Результат accuracy на тестовой выборке равен 0.776, что превышает требование к доле правильных ответов (0.75). Таким образом, цель исследования достигнута.

## Выводы

На первом этапе исследования мы разбили исходную выборку на 3 части в соотношении 3:1:1.
Определили целевой признак (is_ultra) и создали отдельные датасеты для каждой выборки:
* Обучение пройдет на данных из датасета train_data
* Валидация модели на данных из датасета valid_data
* Лучшая модель по валидации будет применена на данных датасета test_data

Протестировали 3 модели Классификации: Решающее дерево, Случайный лес и Логистическая регрессия. 
В ходе исследований получили результаты:

* Точность (accuracy) модели Решающего дерева: 0.785, при max_depth равной 3.
* Точность (accuracy) модели Случайного леса: 0.764, при max_depth равной 5 и n_estimators равному 2.
* Точность (accuracy) модели Логистической регресии: 0.779

Лучший результат показала модель Решающего дерева: 0.785 (max_depth = 3).  
Далее проверили качество этой модели на тестовой выборке. Значение accuracy на тестовой выборке позволяет оценить, насколько хорошо модель обобщает на новых, неизвестных данных.  
Мы получили accuracy равное 0.776, что означает, что наша модель показывает хорошие результаты на новых данных.